# Эксперимент 01: пайплайн фонемного выравнивания речи

Цель: продемонстрировать работу пайплайна на основе `facebook/wav2vec2-lv-60-espeak-cv-ft` —
от подготовки данных TIMIT до инференса, метрик и REST API.

Весь «боевой» код находится в `src/` — этот ноутбук только импортирует и вызывает его,
не дублируя логику (см. `notebooks/README.md`, `src/README.md`).

In [ ]:
import sys
from pathlib import Path

# Добавляем корень проекта в sys.path, чтобы работали импорты `from src...`
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, resolve_path
from src.env_setup import apply_environment
from src.logging_utils import setup_logger

logger = setup_logger()
cfg = load_config()
apply_environment(cfg, logger=logger)
cfg

## 1. Подготовка данных (TIMIT)

Этот блок скачивает датасет TIMIT с Kaggle и конвертирует его в единый формат
`audio/*.wav` + `annotations/*.json`. Требует заполненного `configs/.env`
(см. `configs/README.md`). По умолчанию выключен (`RUN_DOWNLOAD = False`),
чтобы ноутбук можно было запускать без Kaggle credentials.

In [ ]:
RUN_DOWNLOAD = True  # поставь True, когда норм настроены configs/.env

# Количество файлов для EDA и обработки.
# 200–500 обеспечивают покрытие всех 8 диалектов TIMIT при случайной выборке.
EDA_SAMPLE_SIZE = 300  # можно изменить в диапазоне 200–500

if RUN_DOWNLOAD:
    from src.data.download_timit import download_and_extract_timit, load_kaggle_credentials
    from src.data.build_dataset import build_dataset

    load_kaggle_credentials(env_path=str(resolve_path('configs/.env')), logger=logger)
    extract_root = download_and_extract_timit(resolve_path(cfg['paths']['dataset_root']), logger=logger)
    stats = build_dataset(
        extract_root,
        resolve_path(cfg['paths']['audio_dir']),
        resolve_path(cfg['paths']['annotations_dir']),
        max_files=EDA_SAMPLE_SIZE,   # случайная выборка задаётся внутри build_dataset
        random_sample=True,           # перемешать перед нарезкой → репрезентативность диалектов
        random_seed=42,
        logger=logger,
    )
    print(stats)
else:
    EDA_SAMPLE_SIZE = 300
    print('Пропускаем скачивание — RUN_DOWNLOAD=False')


## 2. Разведочный анализ: визуализация аудио

Случайные образцы из `data/processed/input_audio/audio`: осциллограмма, спектр (БПФ),
мел-спектрограмма.

In [ ]:
from src.data.audio_utils import list_audio_files
from src.visualize import visualize_samples

audio_dir = resolve_path(cfg['paths']['audio_dir'])
audio_files = list_audio_files(audio_dir, cfg['audio']['valid_audio_extensions'])
print(f'Найдено аудиофайлов: {len(audio_files)}')

visualize_samples(audio_files, str(audio_dir), num_samples=3)

## 2.1 EDA: распределения по датасету

Анализ всего набора файлов TIMIT: длительности, количество фонем на файл,
частоты встречаемости фонем, распределение по диалектам и спикерам.
Выполняется через модуль `src.eda`.

In [ ]:
from src.eda import (
    compute_dataset_stats,
    plot_duration_distribution,
    plot_phoneme_frequency,
    plot_dialect_distribution,
    print_eda_summary,
)
import json
from pathlib import Path

audio_dir = resolve_path(cfg['paths']['audio_dir'])
annotations_dir = resolve_path(cfg['paths']['annotations_dir'])
artifacts_dir = resolve_path(cfg['paths']['artifacts_dir'])
artifacts_dir.mkdir(parents=True, exist_ok=True)

# Собираем статистику по всем файлам датасета
stats = compute_dataset_stats(audio_dir, annotations_dir, logger=logger)
print(f"Всего файлов в датасете: {stats['total_files']}")
print(f"Файлов с аннотациями: {stats['files_with_annotations']}")
print(f"Средняя длительность: {stats['mean_duration_sec']:.2f} с")
print(f"Всего уникальных фонем: {stats['unique_phonemes']}")


In [ ]:
# График 1: Распределение длительностей аудио
plot_duration_distribution(stats, save_path=artifacts_dir / 'eda_duration_dist.png')


In [ ]:
# График 2: Топ-20 самых частых фонем
plot_phoneme_frequency(stats, top_n=20, save_path=artifacts_dir / 'eda_phoneme_freq.png')


In [ ]:
# График 3: Распределение по диалектам и количество фонем на файл
plot_dialect_distribution(stats, save_path=artifacts_dir / 'eda_dialect_dist.png')


In [ ]:
# Текстовые выводы по EDA
print_eda_summary(stats)


## 2.2 EDA: метрики по всему датасету

Запускаем пайплайн по всем файлам и строим сводные графики метрик качества:
гистограмма PER, boxplot ошибок границ, таблица топ-5 худших файлов по PER.
Если пайплайн уже запускался, подгружаем готовый `metrics_summary.json`.

In [ ]:
from src.eda import (
    plot_per_histogram,
    plot_boundary_errors_boxplot,
    print_worst_files_table,
)
from src.models.inference import extract_and_save_phonemes
from src.models.metrics import compute_metrics, summarize_metric_rows
from src.models.model_loader import load_model
from src.features.annotation_loader import load_reference_annotation
import json as _json

output_dir    = resolve_path(cfg['paths']['output_dir'])
artifacts_dir = resolve_path(cfg['paths']['artifacts_dir'])
annotations_dir = resolve_path(cfg['paths']['annotations_dir'])
artifacts_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

# audio_files уже определён выше (ячейка 2.1) — это все файлы из audio_dir
# После build_dataset с EDA_SAMPLE_SIZE=300 там лежит 200–500 файлов.
print(f'Файлов в audio_dir для прогона метрик: {len(audio_files)}')

# ── Прогон пайплайна ──────────────────────────────────────────────────────
# Если metrics_summary.json уже существует и содержит нужное количество файлов,
# пересчёт пропускается. Поставь RUN_FULL_PIPELINE=True чтобы пересчитать.
RUN_FULL_PIPELINE = False

metrics_json = output_dir / 'metrics_summary.json'
per_file_rows = []

if not RUN_FULL_PIPELINE and metrics_json.exists():
    raw = _json.loads(metrics_json.read_text(encoding='utf-8'))
    # Файл может быть list[dict] (build_metrics_table) или {per_file:[...]} (run_pipeline)
    if isinstance(raw, list):
        per_file_rows = [r for r in raw if r.get('file_id') != 'AVERAGE']
    else:
        per_file_rows = raw.get('per_file', [])

# Если строк меньше половины audio_files — запускаем полный прогон
if len(per_file_rows) < len(audio_files) // 2:
    print(f'Метрик в кэше: {len(per_file_rows)}, файлов: {len(audio_files)} — запускаем прогон...')

    frame_duration  = cfg['model']['frame_duration']
    top_db          = cfg['model']['top_db']
    tolerance_ms    = tuple(cfg['processing']['tolerance_ms'])
    valid_ann_ext   = cfg['audio']['valid_annotation_extensions']
    valid_audio_ext = cfg['audio']['valid_audio_extensions']

    # Загружаем модель (если ещё не загружена в ячейке 3)
    if 'processor' not in dir() or processor is None:
        processor, model, pad_token_id = load_model(cfg['model']['model_id'], logger=logger)

    per_file_rows = []
    for filepath in audio_files:
        rel   = __import__('os').path.relpath(filepath, str(audio_dir))
        bname = __import__('os').path.splitext(rel)[0].replace(__import__('os').sep, '__')
        out_sub = str(output_dir / bname)
        try:
            meta = extract_and_save_phonemes(
                filepath, out_sub, model, processor, pad_token_id,
                audio_dir=str(audio_dir),
                annotations_dir=str(annotations_dir),
                valid_annotation_extensions=valid_ann_ext,
                frame_duration=frame_duration,
                top_db=top_db,
            )
            pred_segs = meta['prediction']['segments']
            ref_segs  = meta['reference']['segments']
            m = compute_metrics(pred_segs, ref_segs, tolerance_ms=tolerance_ms)
            per_file_rows.append({'source_file': meta['source_file'], **m})
        except Exception as _e:
            logger.error(f'Ошибка {rel}: {_e}')

    # Сохраняем в формате {per_file: [...], summary: {...}}
    summary = summarize_metric_rows(per_file_rows)
    metrics_json.write_text(
        _json.dumps({'per_file': per_file_rows, 'summary': summary},
                    ensure_ascii=False, indent=2),
        encoding='utf-8'
    )
    print(f'Прогон завершён. Обработано файлов: {len(per_file_rows)}')
else:
    print(f'Загружено из кэша: {len(per_file_rows)} файлов')

print(f'Итого строк с метриками: {len(per_file_rows)}')


In [ ]:
# График 4: Гистограмма PER по файлам
plot_per_histogram(per_file_rows, save_path=artifacts_dir / 'eda_per_histogram.png')


In [ ]:
# График 5: Boxplot ошибок границ
plot_boundary_errors_boxplot(per_file_rows, save_path=artifacts_dir / 'eda_boundary_boxplot.png')


In [ ]:
# Таблица: топ-5 худших файлов по PER
print_worst_files_table(per_file_rows, top_n=5)


## 3. Загрузка модели

`facebook/wav2vec2-lv-60-espeak-cv-ft` — модель фонемного распознавания (CTC).

In [ ]:
from src.models.model_loader import load_model

processor, model, pad_token_id = load_model(cfg['model']['model_id'], logger=logger)

## 4. Прогон пайплайна на одном файле (демо)

Использует `src.models.inference.extract_and_save_phonemes` — извлекает фонемы,
корректирует границы и сравнивает с эталонной аннотацией (если найдена).

In [ ]:
from src.models.inference import extract_and_save_phonemes
from src.models.metrics import compute_metrics

if audio_files:
    sample_file = audio_files[0]
    output_subfolder = resolve_path(cfg['paths']['output_dir']) / 'demo_sample'

    metadata = extract_and_save_phonemes(
        sample_file,
        str(output_subfolder),
        model,
        processor,
        pad_token_id,
        audio_dir=str(audio_dir),
        annotations_dir=str(resolve_path(cfg['paths']['annotations_dir'])),
        valid_annotation_extensions=cfg['audio']['valid_annotation_extensions'],
        frame_duration=cfg['model']['frame_duration'],
        top_db=cfg['model']['top_db'],
    )

    metrics = compute_metrics(
        metadata['prediction']['segments'],
        metadata['reference']['segments'],
        tolerance_ms=tuple(cfg['processing']['tolerance_ms']),
    )
    print(f"Извлечено фонем: {metadata['prediction']['count']}")
    print(f"PER: {metrics.get('per')}, F1: {metrics.get('phoneme_f1')}")
else:
    print('Нет аудиофайлов для демонстрации — сначала выполните подготовку данных (шаг 1).')

## 5. Полный batch-пайплайн по всем файлам

Эквивалент команды `python -m src.run_pipeline`: обрабатывает все файлы в `audio_dir`,
сохраняет `metadata.json`, сводные метрики `metrics_summary.json` и таблицу
`metrics_comparison.csv` в `output_dir`.

In [ ]:
from src import run_pipeline

# run_pipeline.main() сам загружает конфиг и модель — для демонстрации в ноутбуке
# можно вызвать его напрямую:
# run_pipeline.main()
print('Чтобы запустить полный пайплайн, выполните: run_pipeline.main()')

## 6. REST API сервис

Поднимает `/health` и `/predict` в фоновом потоке, не блокируя ноутбук.

In [ ]:
RUN_SERVICE = True  # поставьте True, чтобы поднять сервис прямо из ноутбука

if RUN_SERVICE:
    from src.service.api import run_in_background
    run_in_background()
else:
    print('Сервис не запущен. Запустить из терминала: python -m src.service.api')

## 7. Сравнение: Baseline vs Improved (критерий 4)

**Вариант B**: одна модель (`facebook/wav2vec2-lv-60-espeak-cv-ft`), два режима обработки:

| Вариант | Описание |
|---|---|
| **Baseline** | Сырые CTC-предсказания без постобработки |
| **Improved** | CTC + акустическая коррекция границ (`librosa.effects.split/trim`) |

Единственное отличие — наша постобработка в `correct_segment_boundaries()`.
Оба варианта вычисляются за **один проход** через модель (экономия памяти и времени).

In [ ]:
from src.compare_models import (
    run_both_variants,
    build_comparison_table,
    plot_comparison,
    plot_boundary_accuracy_by_dialect,
)

audio_dir  = resolve_path(cfg['paths']['audio_dir'])
output_dir = resolve_path(cfg['paths']['output_dir'])
artifacts_dir = resolve_path(cfg['paths']['artifacts_dir'])

audio_files = list_audio_files(audio_dir, cfg['audio']['valid_audio_extensions'])
print(f'Аудиофайлов для сравнения: {len(audio_files)}')


In [ ]:
# Прогон обоих вариантов (один проход модели на каждый файл)
RUN_COMPARISON = True  # поставь True для первого запуска
comparison_json = output_dir / 'model_comparison.json'

if RUN_COMPARISON or not comparison_json.exists():
    baseline_rows, improved_rows = run_both_variants(
        audio_files, output_dir, cfg, logger
    )
else:
    # Загружаем уже посчитанные результаты
    import pandas as pd
    print('Загружаем готовые результаты из', comparison_json)
    cmp_df = pd.read_json(comparison_json)
    print(cmp_df.to_string())


In [ ]:
# Таблица сравнения + графики
try:
    _ = baseline_rows
    has_rows = bool(baseline_rows)
except NameError:
    has_rows = False

if has_rows:
    comparison_df = build_comparison_table(baseline_rows, improved_rows, output_dir, logger)

    # График 1: PER / Boundary Exact Accuracy @20ms & @50ms / IoU
    plot_comparison(comparison_df, output_dir, logger)

    # График 2: Boundary Exact Accuracy @20ms и @50ms по диалектам TIMIT
    plot_boundary_accuracy_by_dialect(baseline_rows, improved_rows, output_dir, logger)

    # Копируем графики в artifacts/
    import shutil
    for png in ['model_comparison_plot.png', 'boundary_accuracy_by_dialect.png']:
        src_path = output_dir / png
        if src_path.exists():
            shutil.copy(src_path, artifacts_dir / png)
    print('Артефакты сохранены в', artifacts_dir)


### Интерпретация результатов сравнения

По итогам прогона на 150 файлах (`files_processed = 150`) получены следующие значения:

| Метрика | baseline | improved | Δ (improved − baseline) |
|---|---|---|---|
| **PER** | 0.6987 | 0.6987 | 0.0000 |
| **Boundary Exact Accuracy @20 ms** | 0.2847 | 0.2795 | −0.0052 |
| **Boundary Exact Accuracy @50 ms** | 0.7601 | 0.8068 | **+0.0467** |
| **Segment IoU mean** | 0.3267 | 0.4671 | **+0.1404** |

- **PER (Phoneme Error Rate)** не изменился — и это ожидаемо: акустическая коррекция границ
  (`librosa.effects.split/trim`) трогает только тайминги начала/конца сегментов, а не сама
  последовательность распознанных фонем, на которой считается PER.
- **Boundary Exact Accuracy @20 ms** у `improved` чуть ниже, чем у `baseline` (0.2795 против 0.2847).
  То есть для очень строгого допуска в 20 мс коррекция границ местами «передвигает» границу дальше
  от истинной — небольшая, но реальная просадка точности на самом мелком масштабе.
- **Boundary Exact Accuracy @50 ms**, напротив, заметно выросла (0.8068 против 0.7601, +0.047) —
  при более мягком допуске коррекция чаще попадает в нужный диапазон, чем сырые CTC-границы.
- **Segment IoU mean** улучшилась сильнее всего (0.4671 против 0.3267, +0.14) — это говорит о том,
  что предсказанные интервалы фонем в среднем куда точнее перекрывают эталонные, даже если точное
  совпадение границ в пределах 20 мс не всегда достигается.

> **Вывод**: акустическая постобработка не улучшает всё одновременно. Она даёт явный выигрыш по
> IoU и по допуску 50 мс (границы становятся точнее «в среднем»), но слегка проигрывает на самом
> строгом допуске 20 мс — вероятно, из-за того, что librosa-коррекция иногда сдвигает границу
> сильнее, чем нужно для попадания в самый узкий интервал. PER при этом не меняется, что
> подтверждает: коррекция влияет только на тайминги, а не на распознанные фонемы. Итог — это
> компромисс, а не безусловное улучшение по всем метрикам сразу.
